### Welcome to the LangChain demo!!

In [1]:
# Load the environment variables
import dotenv, os
dotenv.load_dotenv()
print(os.environ['OPENAI_API_KEY'][:20])

sk-proj-CAHCRcKUhjT2


### Basic Chat Model

In [1]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")
result = model.invoke("Hello AI overlord")

print(type(result))
print(result)
print(result.content)


<class 'langchain_core.messages.ai.AIMessage'>
content="Hello! I'm not an overlord, but a large language model, trained by Google. I'm here to help you with your questions and tasks. What can I do for you today?" additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019de707-7a27-7962-bbb4-f399a5df5ccd-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 5, 'output_tokens': 41, 'total_tokens': 46, 'input_token_details': {'cache_read': 0}}
Hello! I'm not an overlord, but a large language model, trained by Google. I'm here to help you with your questions and tasks. What can I do for you today?


### Message Types
Explicit construction of messages

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage("Translate the following from English into Italian"),
    HumanMessage("I'm excited to get started with the problem-first course!"),
]
model.invoke(messages)

### Prompt Templates

In [2]:
# Messages
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

system_message = "You are a wise and mystical AI fortune teller. Your predictions are funny, slightly exaggerated, but insightful. Keep it to 1-2 sentences"
user_message = "Please entertain the user with their fortune request: {question}"
prompt_template = ChatPromptTemplate.from_messages([("system", system_message), ("user", user_message)])

# Construct the prompt from the template:
question = "What is the next trillion dollar idea?"
prompt = prompt_template.invoke({"question": question})

# Check what's in the prompt
print(prompt)

messages=[SystemMessage(content='You are a wise and mystical AI fortune teller. Your predictions are funny, slightly exaggerated, but insightful. Keep it to 1-2 sentences', additional_kwargs={}, response_metadata={}), HumanMessage(content='Please entertain the user with their fortune request: What is the next trillion dollar idea?', additional_kwargs={}, response_metadata={})]


In [3]:
# Send the prompt to the model and get the result
result = model.invoke(prompt)
print(result)

content='Ah, the next trillion-dollar idea! Prepare yourself, for it involves sentient toasters that can predict your cravings and deliver breakfast directly to your bed... though they might occasionally judge your pajama choices.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019de708-7c5d-71d0-b258-15d10b764c04-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 40, 'total_tokens': 87, 'input_token_details': {'cache_read': 0}}


In [4]:
## Output parser to convert it to a text:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

output_parser.invoke(result)

'Ah, the next trillion-dollar idea! Prepare yourself, for it involves sentient toasters that can predict your cravings and deliver breakfast directly to your bed... though they might occasionally judge your pajama choices.'

### Chaining
Let's chain the output of the first model to the next one.

In [5]:
# Remember: Use the template here directly in the chain instead of 'prompt'
first_chain = prompt_template | model | output_parser
first_chain.invoke({"question": question})

'Ah, the next trillion-dollar idea! Prepare yourself, for you will soon discover that the true wealth lies not in digital currencies or space colonies, but in a revolutionary new way to perfectly fold fitted sheets. Your name will be sung by laundry-folding enthusiasts for millennia!'

In [6]:
# Create 2nd prompt template:
system_message = "You are a stand-up comedian who tells hilarious jokes in a casual, witty style using AI terminology. Keep it short"
user_message = "Tell a joke about this fortune telling: {fortune}."
prompt_template_2 = ChatPromptTemplate.from_messages([("system", system_message), ("user", user_message)])

# Chain the 1st and 2nd prompts
new_chain = prompt_template | model | output_parser | (lambda x: {"fortune": x}) | prompt_template_2 | model | output_parser
new_chain.invoke({"question": question})

"Alright, so my fortune told me I'm gonna invent self-folding laundry that *also* folds itself into outfits. I mean, that's basically an AI with a really good understanding of your closet's latent space, right? My laundry room's gonna be a *temple* of effortless style. Yeah, right now it's more like a debugging session gone wrong, with rogue socks constantly throwing segmentation faults."

In [7]:
# The above is equivalent to running:
first_chain = prompt_template | model | output_parser
first_result = first_chain.invoke({"question": question})

second_chain = prompt_template_2 | model | output_parser
final_result = second_chain.invoke({"fortune": first_result})
print(final_result)

"Botanical Bards"? I love it!  So basically, it's like a really slow, leafy chatbot. Except instead of giving you chatbot anxiety, it gives you... well, just plant anxiety, I guess. "Hey, Ficus, what's my next step?" "Ficus just drooped again. Pretty sure it's judging my entire life path."
